# Setup workflow environment

In [14]:
using Pkg; Pkg.activate(dirname(Base.current_project()))

import TulipaIO as TIO
import TulipaEnergyModel as TEM
using DuckDB
using DataFrames
using HiGHS

# For Win. system to fix the KaTex parse error in Jupyter Notebook
Base.show(stdout, ::MIME"text/latex", df::DataFrame) = show(stdout, MIME("text/plain"), df)

#= For utility functions
    - print_annual_total_prod(DBconnection, years...)
    - inv_cost, fom_cost, var_cost = objective_terms_value(TulipaProblem, DBconnection)
    - annual_flows_between_assets(DBconnection, from_asset_term, to_asset_term)
=#
include("../src/util_analysis.jl");

  Activating project at `c:\ModellingRepos\VintageDemo`


# Multi-year investment model instances

- Milestone years: 2030, 2040 and 2050.
- The system has 30GW initial wind capacity built in 2025, the model can choose to invest in wind in three milestone years: 2030, 2040, 2050.

## Instance 1. No technology vintage of units

> Note: the `output_dir` must exist/(be created) beforehand 

### 1.1 Build and run the model instance

In [15]:
instance = "1-no-vintage"
# Define and build the input output directories
input_dir = "model-instance-Tulipa/input-$(instance)"
output_dir = joinpath(@__DIR__, "model-instance-Tulipa/output-$(instance)")

# Always build a new result directory
rm(output_dir, force=true, recursive=true) 
mkdir(output_dir);

# Create the connection and prepare input data
connection_no_vintage = DBInterface.connect(DuckDB.DB)
TIO.read_csv_folder(connection_no_vintage, input_dir)
TEM.populate_with_defaults!(connection_no_vintage)

_connection = connection_no_vintage

DuckDB.DB(":memory:")

In [16]:
# Set the log file paths
model_log  = joinpath(@__DIR__, "model-instance-Tulipa/model-log_$(instance).txt")
solver_log = joinpath(@__DIR__, "model-instance-Tulipa/temp_solver-log_$(instance).txt")  # temp, to be folded into model_log below

# Run the model instance
ep = multiyear_no_vintage = TEM.run_scenario(
    _connection;
    optimizer = HiGHS.Optimizer,
    optimizer_parameters = Dict(
        "output_flag" => true,      # TEM.default_parameters(HiGHS.Optimizer): Dict("output_flag" => false)
        "log_file"    => solver_log,  # HiGHS writes its full solver log (temp) here (still shown live)
        "mip_rel_gap"  => 0.0,      # default is 1e-4 (0.01%)
        "mip_abs_gap"  => 0.0,      # default is 1e-6
    ), 
    output_folder = output_dir, 
    # model_file_name = joinpath(output_dir, "model.lp"),
    show_log=true,
    # Always run with this option in a new or restarted kernel to ensure the time is dedicated to this solve only.
    log_file = model_log,   # TimerOutputs table only records timing information (written by run_scenario)
)

# Append EnergyProblem summary + full solver log into model_log; remove the temp solver log
append_run_report!(ep, model_log; solver_log = solver_log)

ep   # keep the EnergyProblem instance as the cell's displayed output

Running HiGHS 1.15.0 (git hash: 8396001901): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 341640 rows; 315363 cols; 683271 nonzeros; 3 integer variables (0 binary)
Coefficient ranges:
  Matrix  [2e-04, 1e+00]
  Cost    [1e-03, 3e+03]
  Bound   [1e+02, 2e+02]
  RHS     [5e-03, 9e+01]
Presolving model
26276 rows, 123335 cols, 175882 nonzeros 0s
23945 rows, 120808 cols, 168922 nonzeros 2s
Presolve reductions: rows 23945(-317695); columns 120808(-194555); nonzeros 168922(-514349) 

Solving MIP model with:
   23945 rows
   120808 cols (0 binary, 3 integer, 0 implied int., 120805 continuous, 0 domain fixed)
   168922 nonzeros
   Thread count 4 (of 8 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T 

EnergyProblem:
  - Model created!
    - Number of variables: 315363
    - Number of constraints for variable bounds: 315363
    - Number of structural constraints: 341640
  - Model solved!
    - Termination status: OPTIMAL
    - Objective value: 797099.5238131845
    - Objective breakdown:
      - assets_fixed_cost_aggregated_vintage_method: 134879.9169460226
      - assets_fixed_cost_compact_vintage_method: 0.0
      - assets_investment_cost: 255372.26939289505
      - flows_fixed_cost: 0.0
      - flows_investment_cost: 0.0
      - flows_operational_cost: 406847.33747426164
      - storage_assets_energy_fixed_cost: 0.0
      - storage_assets_energy_investment_cost: 0.0
      - units_on_operational_cost: 0.0
      - vintage_flows_operational_cost: 0.0


### 1.2 Key results

#### Capacity

- In this case, we will not be able to differentiate assets built within the same milestone year, they will simply be considered the same as the units built in the milestone year.

In [17]:
function print_capacity_investment(f::Function, DB_conn::DuckDB.DB)
    # initial wind capacity
    println(
        "Initial wind capacity (GW): ", 
        filter(row -> f(row) && row.initial_units > 0.0, TIO.get_table(DB_conn, "asset_both")[
            :, [:asset, :milestone_year, :commission_year, :decommissionable, :initial_units]
        ]), 
        "\n")
    # invested capacity
    println(
        "Invested wind capacity (GW): # of rows = # of investment variables.\n", 
        filter(f, TIO.get_table(DB_conn, "var_assets_investment")[
            :, [:asset, :milestone_year, :investment_integer, :capacity, :investment_limit, :solution]
        ]), 
        "\n")
end

print_capacity_investment(row -> row.asset=="wind", _connection)

Initial wind capacity (GW): 3×5 DataFrame
 Row │ asset   milestone_year  commission_year  decommissionable  initial_units 
     │ String  Int32           Int32            Bool              Float64       
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ wind              2030             2030             false           30.0
   2 │ wind              2040             2040             false           30.0
   3 │ wind              2050             2050             false           30.0

Invested wind capacity (GW): # of rows = # of investment variables.
3×6 DataFrame
 Row │ asset   milestone_year  investment_integer  capacity  investment_limit  solution 
     │ String  Int32           Bool                Float64   Float64           Float64  
─────┼──────────────────────────────────────────────────────────────────────────────────
   1 │ wind              2030                true       1.0           107.567     107.0
   2 │ wind              2040        

#### Annual productions & total system cost

In [18]:
function print_annual_total_prod(DB_conn::DuckDB.DB, milestone_years::Int...)
    for year in milestone_years
        println(year, "s")
        filter(
            row -> row.milestone_year == year, 
            annual_flows_between_assets(DB_conn, "wind", "demand")
        ) |> row -> println("\t wind prodution: $(annual_total_prods(DB_conn, "wind", "demand", year)) TWh p.a.")
        filter(
            row -> row.milestone_year == year, 
            annual_flows_between_assets(DB_conn, "ens", "demand")
        ) |> row -> println("\t market supply: $(annual_total_prods(DB_conn, "ens", "demand", year)) TWh p.a.")
    end
end

print_annual_total_prod(_connection, 2030, 2040, 2050)

total_cost = ep.objective_value
println("Total system cost: $(round(total_cost/1000, digits=2)) Billion €")

inv_cost, fixed_om_cost, variable_om_cost = objective_terms_value(ep, _connection)
println(
    "\t investment: $(round(inv_cost/1000, digits=2)) Billion € \n", 
    "\t fixed O&M: $(round(fixed_om_cost/1000, digits=2)) Billion € \n",
    "\t variable O&M: $(round(variable_om_cost/1000, digits=2)) Billion €"
)
println(
    "Total system cost ≈ investment + fixed O&M + variable O&M: ", 
    total_cost ≈ inv_cost + fixed_om_cost + variable_om_cost
)
# @assert total_cost ≈ inv_cost + fixed_om_cost + variable_om_cost

2030s
	 wind prodution: 605.61 TWh p.a.
	 market supply: 207.05 TWh p.a.
2040s
	 wind prodution: 858.42 TWh p.a.
	 market supply: 271.03 TWh p.a.
2050s
	 wind prodution: 841.72 TWh p.a.
	 market supply: 388.51 TWh p.a.
Total system cost: 797.1 Billion €
	 investment: 255.37 Billion € 
	 fixed O&M: 134.88 Billion € 
	 variable O&M: 406.85 Billion €
Total system cost ≈ investment + fixed O&M + variable O&M: true


## Instance 2. Explicit technology vintage of units - standard method

- Difference from **Instance 1**: `asset.csv`, `asset-milestone.csv`, `asset-commission.csv`, `asset-both`, `assets-profiles.csv`, `flow*.csv`
- In any milestone year, only the tech. vintage of the same year is available for investment (explicitly regulated by setting the `investable` parameter in `asset-milestone.csv`)

### 2.1 Build and run the model instance

In [19]:
instance = "2-vintage-standard"
# Define and build the input output directories
input_dir = "model-instance-Tulipa/input-$(instance)"
output_dir = joinpath(@__DIR__, "model-instance-Tulipa/output-$(instance)")

# Always build a new result directory
rm(output_dir, force=true, recursive=true) 
mkdir(output_dir);

# Create the connection and prepare input data
connection_vintage_standard = DBInterface.connect(DuckDB.DB)
TIO.read_csv_folder(connection_vintage_standard, input_dir)
TEM.populate_with_defaults!(connection_vintage_standard)

_connection = connection_vintage_standard

DuckDB.DB(":memory:")

In [20]:
# Set the log file paths
model_log  = joinpath(@__DIR__, "model-instance-Tulipa/model-log_$(instance).txt")
solver_log = joinpath(@__DIR__, "model-instance-Tulipa/temp_solver-log_$(instance).txt")  # temp, to be folded into model_log below

# Run the model instance
ep = multiyear_vintage_standard = TEM.run_scenario(
    _connection;
    optimizer = HiGHS.Optimizer,
    optimizer_parameters = Dict(
        "output_flag" => true,      # TEM.default_parameters(HiGHS.Optimizer): Dict("output_flag" => false)
        "log_file"    => solver_log,  # HiGHS writes its full solver log (temp) here (still shown live)
        "mip_rel_gap"  => 0.0,      # default is 1e-4 (0.01%)
        "mip_abs_gap"  => 0.0,      # default is 1e-6
    ), 
    output_folder = output_dir, 
    # model_file_name = joinpath(output_dir, "model.lp"),
    show_log=true,
    # Always run with this option in a new or restarted kernel to ensure the time is dedicated to this solve only.
    log_file = model_log,   # TimerOutputs table only records timing information (written by run_scenario)
)

# Append EnergyProblem summary + full solver log into model_log; remove the temp solver log
append_run_report!(ep, model_log; solver_log = solver_log)

ep   # keep the EnergyProblem instance as the cell's displayed output

Running HiGHS 1.15.0 (git hash: 8396001901): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 420480 rows; 788403 cols; 1629351 nonzeros; 3 integer variables (0 binary)
Coefficient ranges:
  Matrix  [3e-06, 1e+00]
  Cost    [1e-03, 3e+03]
  Bound   [1e+02, 2e+02]
  RHS     [4e-04, 9e+01]
Presolving model
236511 rows, 630669 cols, 1156203 nonzeros 2s
228279 rows, 611123 cols, 1121782 nonzeros 6s
Presolve reductions: rows 228279(-192201); columns 611123(-177280); nonzeros 1121782(-507569) 

Solving MIP model with:
   228279 rows
   611123 cols (0 binary, 3 integer, 0 implied int., 611120 continuous, 0 domain fixed)
   1121782 nonzeros
   Thread count 4 (of 8 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Sol

EnergyProblem:
  - Model created!
    - Number of variables: 788403
    - Number of constraints for variable bounds: 788403
    - Number of structural constraints: 420480
  - Model solved!
    - Termination status: OPTIMAL
    - Objective value: 771132.0135663874
    - Objective breakdown:
      - assets_fixed_cost_aggregated_vintage_method: 152461.80436257186
      - assets_fixed_cost_compact_vintage_method: 0.0
      - assets_investment_cost: 291002.23095674335
      - flows_fixed_cost: 0.0
      - flows_investment_cost: 0.0
      - flows_operational_cost: 327667.97824708046
      - storage_assets_energy_fixed_cost: 0.0
      - storage_assets_energy_investment_cost: 0.0
      - units_on_operational_cost: 0.0
      - vintage_flows_operational_cost: 0.0


### 2.2 Key results

#### Capacity

- **Assumption**:
    - assets of a vintage is exclusively investable in the same milestone year as the vintage indicated by the asset name: see parameter `investable` in `asset-milestone.csv`.

In [21]:
print_capacity_investment(row -> occursin("wind", row.asset), _connection)

Initial wind capacity (GW): 3×5 DataFrame
 Row │ asset   milestone_year  commission_year  decommissionable  initial_units 
     │ String  Int32           Int32            Bool              Float64       
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ wind25            2030             2030             false           30.0
   2 │ wind25            2040             2040             false           30.0
   3 │ wind25            2050             2050             false           30.0

Invested wind capacity (GW): # of rows = # of investment variables.
3×6 DataFrame
 Row │ asset   milestone_year  investment_integer  capacity  investment_limit  solution 
     │ String  Int32           Bool                Float64   Float64           Float64  
─────┼──────────────────────────────────────────────────────────────────────────────────
   1 │ wind30            2030                true       1.0           107.567     107.0
   2 │ wind40            2040        

#### Annual productions & total system cost

In [22]:
print_annual_total_prod(_connection, 2030, 2040, 2050)

total_cost = ep.objective_value
println("Total system cost: $(round(total_cost/1000, digits=2)) Billion €")

inv_cost, fixed_om_cost, variable_om_cost = objective_terms_value(ep, _connection)
println(
    "\t investment: $(round(inv_cost/1000, digits=2)) Billion € \n", 
    "\t fixed O&M: $(round(fixed_om_cost/1000, digits=2)) Billion € \n",
    "\t variable O&M: $(round(variable_om_cost/1000, digits=2)) Billion €"
)
println(
    "Total system cost ≈ investment + fixed O&M + variable O&M: ", 
    total_cost ≈ inv_cost + fixed_om_cost + variable_om_cost
)
# @assert total_cost ≈ inv_cost + fixed_om_cost + variable_om_cost

2030s
	 wind prodution: 611.93 TWh p.a.
	 market supply: 200.73 TWh p.a.
2040s
	 wind prodution: 954.75 TWh p.a.
	 market supply: 174.69 TWh p.a.
2050s
	 wind prodution: 1022.7 TWh p.a.
	 market supply: 207.53 TWh p.a.
Total system cost: 771.13 Billion €
	 investment: 291.0 Billion € 
	 fixed O&M: 152.46 Billion € 
	 variable O&M: 327.67 Billion €
Total system cost ≈ investment + fixed O&M + variable O&M: true


## Instance 3. Explicit technology vintage of units - compact method

- Difference from **Instance 2**: `asset.csv` (only tha parameter `vintage_method`), `asset-commission.csv`, `asset-both`
- In any milestone year, only the tech. vintage of the same year is available for investment (assumed with setting `vintage_method=compact_profiles` for "wind" `asset`)
- Resulted investment in 2050 differ significantly from the no vintage case because the wind of 2020 vintage (wind commissioned in 2020, defined in `asset-commission.csv`) uses the default availability of `1.0` over the milestone year 2050 due to the lack of availability value is given for 2050 for this vintage (`availability-wind2020` records in `profiles-rep-periods.csv`, which is assigned to the wind commissioned in 2020 in `assets-profiles.csv`)

### 3.1 Build and run the model instance

In [23]:
instance = "3-vintage-compact"
# Define and build the input output directories
input_dir = "model-instance-Tulipa/input-$(instance)"
output_dir = joinpath(@__DIR__, "model-instance-Tulipa/output-$(instance)")

# Always build a new result directory
rm(output_dir, force=true, recursive=true) 
mkdir(output_dir);

# Create the connection and prepare input data
connection_vintage_compact = DBInterface.connect(DuckDB.DB)
TIO.read_csv_folder(connection_vintage_compact, input_dir)
TEM.populate_with_defaults!(connection_vintage_compact)

_connection = connection_vintage_compact

DuckDB.DB(":memory:")

In [24]:
# Set the log file paths
model_log  = joinpath(@__DIR__, "model-instance-Tulipa/model-log_$(instance).txt")
solver_log = joinpath(@__DIR__, "model-instance-Tulipa/temp_solver-log_$(instance).txt")  # temp, to be folded into model_log below

# Run the model instance
ep = multiyear_vintage_compact = TEM.run_scenario(
    _connection;
    optimizer = HiGHS.Optimizer,
    optimizer_parameters = Dict(
        "output_flag" => true,      # TEM.default_parameters(HiGHS.Optimizer): Dict("output_flag" => false)
        "log_file"    => solver_log,  # HiGHS writes its full solver log (temp) here (still shown live)
        "mip_rel_gap"  => 0.0,      # default is 1e-4 (0.01%)
        "mip_abs_gap"  => 0.0,      # default is 1e-6
    ), 
    output_folder = output_dir, 
    # model_file_name = joinpath(output_dir, "model.lp"),
    show_log=true,
    # Always run with this option in a new or restarted kernel to ensure the time is dedicated to this solve only.
    log_file = model_log,   # TimerOutputs table only records timing information (written by run_scenario)
)

# Append EnergyProblem summary + full solver log into model_log; remove the temp solver log
append_run_report!(ep, model_log; solver_log = solver_log)

ep   # keep the EnergyProblem instance as the cell's displayed output

Running HiGHS 1.15.0 (git hash: 8396001901): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 341640 rows; 315363 cols; 683271 nonzeros; 3 integer variables (0 binary)
Coefficient ranges:
  Matrix  [3e-06, 1e+00]
  Cost    [1e-03, 3e+03]
  Bound   [1e+02, 2e+02]
  RHS     [4e-04, 9e+01]
Presolving model
26280 rows, 134823 cols, 187371 nonzeros 0s
24946 rows, 132253 cols, 183315 nonzeros 2s
Presolve reductions: rows 24946(-316694); columns 132253(-183110); nonzeros 183315(-499956) 

Solving MIP model with:
   24946 rows
   132253 cols (0 binary, 3 integer, 0 implied int., 132250 continuous, 0 domain fixed)
   183315 nonzeros
   Thread count 4 (of 8 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T 

EnergyProblem:
  - Model created!
    - Number of variables: 315363
    - Number of constraints for variable bounds: 315363
    - Number of structural constraints: 341640
  - Model solved!
    - Termination status: OPTIMAL
    - Objective value: 771132.0135663874
    - Objective breakdown:
      - assets_fixed_cost_aggregated_vintage_method: 0.0
      - assets_fixed_cost_compact_vintage_method: 152461.80436257186
      - assets_investment_cost: 291002.23095674335
      - flows_fixed_cost: 0.0
      - flows_investment_cost: 0.0
      - flows_operational_cost: 327667.9782470715
      - storage_assets_energy_fixed_cost: 0.0
      - storage_assets_energy_investment_cost: 0.0
      - units_on_operational_cost: 0.0
      - vintage_flows_operational_cost: 0.0


### 3.2 Key results

#### Capacity

- **Assumption**:
    - assets of a vintage is exclusively investable in the same milestone year as the commission (vintage) year: see parameter `investable` in `asset-milestone.csv`.

- Units built in different years are explicitly listed, meaning that their corresponding profiles are also considered.

In [25]:
print_capacity_investment(row -> occursin("wind", row.asset), _connection)

Initial wind capacity (GW): 3×5 DataFrame
 Row │ asset   milestone_year  commission_year  decommissionable  initial_units 
     │ String  Int32           Int32            Bool              Float64       
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ wind              2030             2025             false           30.0
   2 │ wind              2040             2025             false           30.0
   3 │ wind              2050             2025             false           30.0

Invested wind capacity (GW): # of rows = # of investment variables.
3×6 DataFrame
 Row │ asset   milestone_year  investment_integer  capacity  investment_limit  solution 
     │ String  Int32           Bool                Float64   Float64           Float64  
─────┼──────────────────────────────────────────────────────────────────────────────────
   1 │ wind              2030                true       1.0           107.567     107.0
   2 │ wind              2040        

#### Annual productions & total system cost

In [26]:
print_annual_total_prod(_connection, 2030, 2040, 2050)

total_cost = ep.objective_value
println("Total system cost: $(round(total_cost/1000, digits=2)) Billion €")

inv_cost, fixed_om_cost, variable_om_cost = objective_terms_value(ep, _connection)
println(
    "\t investment: $(round(inv_cost/1000, digits=2)) Billion € \n", 
    "\t fixed O&M: $(round(fixed_om_cost/1000, digits=2)) Billion € \n",
    "\t variable O&M: $(round(variable_om_cost/1000, digits=2)) Billion €"
)
println(
    "Total system cost ≈ investment + fixed O&M + variable O&M: ", 
    total_cost ≈ inv_cost + fixed_om_cost + variable_om_cost
)
# @assert total_cost ≈ inv_cost + fixed_om_cost + variable_om_cost

2030s
	 wind prodution: 611.93 TWh p.a.
	 market supply: 200.73 TWh p.a.
2040s
	 wind prodution: 954.75 TWh p.a.
	 market supply: 174.69 TWh p.a.
2050s
	 wind prodution: 1022.7 TWh p.a.
	 market supply: 207.53 TWh p.a.
Total system cost: 771.13 Billion €
	 investment: 291.0 Billion € 
	 fixed O&M: 152.46 Billion € 
	 variable O&M: 327.67 Billion €
Total system cost ≈ investment + fixed O&M + variable O&M: true
